In [1]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import pandas as pd

# Load the CRM report
crm_df = pd.read_csv(r"C:\Users\Hp\Downloads\P1G7\P1G7\PerfectPetClub_CRM_report.csv")

# Load the server log
server_df = pd.read_csv(r"C:\Users\Hp\Downloads\P1G7\P1G7\PerfectPetClub_server_log.csv")

# Load the customer survey
survey_df = pd.read_csv(r"C:\Users\Hp\Downloads\P1G7\P1G7\PerfectPetClub_survey.csv")

In [3]:
print("CRM dataset shape:", crm_df.shape)
print("Server log dataset shape:", server_df.shape)
print("Survey dataset shape:", survey_df.shape)

CRM dataset shape: (7585, 4)
Server log dataset shape: (67249, 5)
Survey dataset shape: (221, 5)


In [4]:
print("CRM columns:")
print(crm_df.columns.tolist())

print("\nServer log columns:")
print(server_df.columns.tolist())

print("\nSurvey columns:")
print(survey_df.columns.tolist())

CRM columns:
['customer_id', 'recieved_datatime', 'platform', 'type']

Server log columns:
['datetime', 'session_id', 'ip', 'login', 'user_agent']

Survey columns:
['email', 'reported_income', 'nps_recommend', 'satisfaction_score', 'online_comparison_freq']


In [5]:
# Check missing values in each dataset

print("CRM missing values:")
print(crm_df.isnull().sum())

print("\nServer log missing values:")
print(server_df.isnull().sum())

print("\nSurvey missing values:")
print(survey_df.isnull().sum())

CRM missing values:
customer_id          0
recieved_datatime    0
platform             0
type                 0
dtype: int64

Server log missing values:
datetime      0
session_id    0
ip            0
login         0
user_agent    0
dtype: int64

Survey missing values:
email                     0
reported_income           0
nps_recommend             0
satisfaction_score        0
online_comparison_freq    0
dtype: int64


In [6]:
# Count duplicate rows in each dataset

print("CRM duplicate rows:", crm_df.duplicated().sum())

print("Server log duplicate rows:", server_df.duplicated().sum())

print("Survey duplicate rows:", survey_df.duplicated().sum())

CRM duplicate rows: 0
Server log duplicate rows: 0
Survey duplicate rows: 0


In [7]:
# Display data types of all columns

print("CRM data types:")
print(crm_df.dtypes)

print("\nServer log data types:")
print(server_df.dtypes)

print("\nSurvey data types:")
print(survey_df.dtypes)

CRM data types:
customer_id           int64
recieved_datatime    object
platform             object
type                 object
dtype: object

Server log data types:
datetime      object
session_id    object
ip            object
login         object
user_agent    object
dtype: object

Survey data types:
email                     object
reported_income           object
nps_recommend              int64
satisfaction_score         int64
online_comparison_freq     int64
dtype: object


In [8]:
# Count unique customer IDs in CRM

print(
    "Unique customers in CRM:",
    crm_df["customer_id"].nunique()
)

# Count unique survey respondents

print(
    "Unique survey respondents:",
    survey_df["email"].nunique()
)

Unique customers in CRM: 200
Unique survey respondents: 221


In [9]:
import sqlite3

# Connect to the SQLite database
db_path = r"C:\Users\Hp\Downloads\P1G7\P1G7\PerfectPetClub.db"

connection = sqlite3.connect(db_path)

In [10]:
# Extract customer information
customer_df = pd.read_sql_query(
    "SELECT id, email, subscription_id FROM customer",
    connection
)

# Extract subscription information
subscription_df = pd.read_sql_query(
    "SELECT * FROM subscription",
    connection
)

subscription_type_df = pd.read_sql_query(
    "SELECT * FROM subscription_type",
    connection
)

# Extract sales and payment information
sale_df = pd.read_sql_query(
    "SELECT * FROM sale",
    connection
)

payment_df = pd.read_sql_query(
    "SELECT id, amount, payment_gateway FROM payment",
    connection
)

# Extract product information
sale_product_df = pd.read_sql_query(
    "SELECT * FROM sale_product",
    connection
)

product_df = pd.read_sql_query(
    "SELECT * FROM product",
    connection
)

product_category_df = pd.read_sql_query(
    "SELECT * FROM product_category",
    connection
)

# Extract product reviews
product_review_df = pd.read_sql_query(
    "SELECT * FROM product_review",
    connection
)

In [11]:
# Check the number of rows and columns in each table

database_tables = {
    "Customer": customer_df,
    "Subscription": subscription_df,
    "Subscription Type": subscription_type_df,
    "Sale": sale_df,
    "Payment": payment_df,
    "Sale Product": sale_product_df,
    "Product": product_df,
    "Product Category": product_category_df,
    "Product Review": product_review_df
}

for table_name, dataframe in database_tables.items():
    print(table_name, ":", dataframe.shape)

Customer : (221, 3)
Subscription : (221, 5)
Subscription Type : (3, 3)
Sale : (23188, 6)
Payment : (23188, 3)
Sale Product : (59759, 4)
Product : (194, 4)
Product Category : (26, 2)
Product Review : (10990, 5)


In [12]:
connection.close()

print("Database connection closed successfully!")

Database connection closed successfully!


In [13]:
# Connect to the PerfectPetClub database
conn = sqlite3.connect(r"C:\Users\Hp\Downloads\P1G7\P1G7\PerfectPetClub.db")

print("Database connected successfully!")

Database connected successfully!


In [14]:
# Retrieve all table names from the database

tables_df = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    AND name NOT LIKE 'sqlite_%'
""", conn)

display(tables_df)

,name
0,payment
1,product_category
2,product
3,subscription_type
4,subscription
5,customer
6,sale
7,sale_product
8,product_review


In [15]:
# Inspect the customer table structure

customer_schema = pd.read_sql_query("""
    PRAGMA table_info(customer)
""", conn)

display(customer_schema)

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,1,None,1
1,1,name,TEXT,1,None,0
2,2,email,TEXT,1,None,0
3,3,password_hash,TEXT,1,None,0
4,4,address_line_1,TEXT,1,None,0
5,5,address_line_2,TEXT,0,None,0
6,6,county,TEXT,1,None,0
7,7,postcode,TEXT,1,None,0
8,8,subscription_id,INTEGER,0,None,0


In [16]:
# Inspect foreign-key relationships in important tables

tables_to_check = [
    "customer",
    "subscription",
    "sale",
    "sale_product",
    "product",
    "product_review"
]

for table in tables_to_check:
    print(f"\nForeign keys in {table}:")

    query = f"PRAGMA foreign_key_list({table})"
    foreign_keys = pd.read_sql_query(query, conn)

    display(foreign_keys)


Foreign keys in customer:


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,subscription,subscription_id,id,NO ACTION,NO ACTION,NONE



Foreign keys in subscription:


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,subscription_type,subscription_type_id,id,NO ACTION,NO ACTION,NONE



Foreign keys in sale:


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,payment,payment_id,id,NO ACTION,NO ACTION,NONE
1,1,0,customer,customer_id,id,NO ACTION,NO ACTION,NONE


... (2 outputs left out)


Foreign keys in product:


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,product_category,product_category_id,id,NO ACTION,NO ACTION,NONE



Foreign keys in product_review:


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,sale,sale_id,id,NO ACTION,NO ACTION,NONE
1,1,0,customer,customer_id,id,NO ACTION,NO ACTION,NONE
2,2,0,product,product_id,id,NO ACTION,NO ACTION,NONE


In [17]:
# Extract data from the SQLite database

customer_df = pd.read_sql_query(
    "SELECT * FROM customer", conn
)

subscription_df = pd.read_sql_query(
    "SELECT * FROM subscription", conn
)

subscription_type_df = pd.read_sql_query(
    "SELECT * FROM subscription_type", conn
)

sale_df = pd.read_sql_query(
    "SELECT * FROM sale", conn
)

sale_product_df = pd.read_sql_query(
    "SELECT * FROM sale_product", conn
)

product_review_df = pd.read_sql_query(
    "SELECT * FROM product_review", conn
)

print("Database tables extracted successfully!")

Database tables extracted successfully!


In [18]:
# Check the number of rows and columns extracted

print("Customer:", customer_df.shape)
print("Subscription:", subscription_df.shape)
print("Subscription type:", subscription_type_df.shape)
print("Sales:", sale_df.shape)
print("Sale products:", sale_product_df.shape)
print("Product reviews:", product_review_df.shape)

Customer: (221, 9)
Subscription: (221, 5)
Subscription type: (3, 3)
Sales: (23188, 6)
Sale products: (59759, 4)
Product reviews: (10990, 5)


In [19]:
# Check subscription auto-renewal counts

print("Auto-renewal distribution:")

print(
    subscription_df["auto_renew"].value_counts()
)

Auto-renewal distribution:
auto_renew
1    135
0     86
Name: count, dtype: int64


In [20]:
# Inspect subscription table column names

print("Subscription columns:")
print(subscription_df.columns.tolist())

print("\nSubscription data types:")
print(subscription_df.dtypes)

print("\nSubscription table shape:")
print(subscription_df.shape)

Subscription columns:
['id', 'start_date', 'end_date', 'auto_renew', 'subscription_type_id']

Subscription data types:
id                       int64
start_date              object
end_date                object
auto_renew               int64
subscription_type_id     int64
dtype: object

Subscription table shape:
(221, 5)


In [21]:
# Check subscription table missing values

print("Missing values:")
print(subscription_df.isnull().sum())

# Check duplicate rows
print("\nDuplicate rows:")
print(subscription_df.duplicated().sum())

# Check unique subscription IDs
print("\nUnique subscription IDs:")
print(subscription_df["id"].nunique())

Missing values:
id                      0
start_date              0
end_date                0
auto_renew              0
subscription_type_id    0
dtype: int64

Duplicate rows:
0

Unique subscription IDs:
221


In [22]:
# Check customer-to-subscription relationships

print(
    "Unique customer IDs:",
    customer_df["id"].nunique()
)

print(
    "Unique customer subscription IDs:",
    customer_df["subscription_id"].nunique()
)

print(
    "Unique subscription IDs:",
    subscription_df["id"].nunique()
)

Unique customer IDs: 221
Unique customer subscription IDs: 221
Unique subscription IDs: 221


In [23]:
# Join customer and subscription tables

customer_subscription_df = pd.merge(
    customer_df,
    subscription_df,
    left_on="subscription_id",
    right_on="id",
    how="left",
    suffixes=("_customer", "_subscription"),
    validate="one_to_one"
)

print("Customer-subscription join completed!")

print(
    "Joined dataset shape:",
    customer_subscription_df.shape
)

print(
    "Number of customer records:",
    customer_subscription_df["id_customer"].nunique()
)

Customer-subscription join completed!
Joined dataset shape: (221, 14)
Number of customer records: 221


In [24]:
# Create provisional churn indicator

customer_subscription_df["churn"] = (
    customer_subscription_df["auto_renew"] == 0
).astype(int)

print("Provisional churn distribution:")

print(
    customer_subscription_df["churn"].value_counts()
) 

Provisional churn distribution:
churn
0    135
1     86
Name: count, dtype: int64


In [25]:
# Check sales date column and data types

print("Sales columns:")
print(sale_df.columns.tolist())

print("\nSales data types:")
print(sale_df.dtypes)

Sales columns:
['id', 'order_date', 'expected_date', 'delivered_date', 'customer_id', 'payment_id']

Sales data types:
id                 int64
order_date        object
expected_date     object
delivered_date    object
customer_id        int64
payment_id         int64
dtype: object


In [26]:
# Convert order_date to datetime

sale_df["order_date"] = pd.to_datetime(
    sale_df["order_date"],
    errors="coerce"
)

print("Order dates converted successfully!")

print("Invalid order dates:",
      sale_df["order_date"].isna().sum())

Order dates converted successfully!
Invalid order dates: 0


In [27]:
# Count purchases made by each customer

sales_features = sale_df.groupby("customer_id").agg(
    total_orders=("id", "nunique"),
    first_order_date=("order_date", "min"),
    last_order_date=("order_date", "max")
).reset_index()

print("Sales features created!")
print("Number of customers:", len(sales_features))

Sales features created!
Number of customers: 221


In [28]:
# Check the sales feature dataset

print("Sales feature shape:", sales_features.shape)

print("\nMissing values:")
print(sales_features.isnull().sum())

print("\nSample sales features:")
display(sales_features.head())

Sales feature shape: (221, 4)

Missing values:
customer_id         0
total_orders        0
first_order_date    0
last_order_date     0
dtype: int64

Sample sales features:


,customer_id,total_orders,first_order_date,last_order_date
0,1,20,2026-01-01,2026-01-10 20:43:36.234
1,2,253,2026-01-01,2026-11-09 21:35:20.498
2,3,304,2026-01-01,2026-11-07 21:31:22.450
3,4,1055,2026-01-01,2026-11-09 21:56:40.499
4,5,44,2026-01-01,2026-04-12 19:52:46.436


In [29]:
# Calculate total product quantity for each sale

quantity_per_sale = sale_product_df.groupby("sale_id").agg(
    total_quantity=("quantity", "sum")
).reset_index()

# Connect product quantities to customer sales

sales_with_quantity = pd.merge(
    sale_df[["id", "customer_id"]],
    quantity_per_sale,
    left_on="id",
    right_on="sale_id",
    how="left",
    validate="one_to_one"
)

# Calculate total product quantity per customer

quantity_features = sales_with_quantity.groupby(
    "customer_id"
).agg(
    total_products_purchased=("total_quantity", "sum")
).reset_index()

print("Purchase quantity features created!")

display(quantity_features.head())

Purchase quantity features created!


,customer_id,total_products_purchased
0,1,46
1,2,748
2,3,824
3,4,7362
4,5,116


In [30]:
# Save intermediate feature datasets

sales_features.to_csv(
    rf"C:\Users\Hp\Downloads\P1G7\P1G7\data processed\sales_features.csv",
    index=False
)

quantity_features.to_csv(
    rf"C:\Users\Hp\Downloads\P1G7\P1G7\data processed\quantity_features.csv",
    index=False
)

print("Sales feature files saved successfully!")

Sales feature files saved successfully!


In [31]:
# Inspect payment table columns

payment_df = pd.read_sql_query(
    "SELECT * FROM payment",
    conn
)

print("Payment table shape:", payment_df.shape)
print("Payment columns:", payment_df.columns.tolist())
print("\nPayment data types:")
print(payment_df.dtypes)

Payment table shape: (23188, 4)
Payment columns: ['id', 'amount', 'payment_gateway', 'transaction_reference']

Payment data types:
id                        int64
amount                    int64
payment_gateway          object
transaction_reference    object
dtype: object


In [32]:
# Join sales with payment information

sales_payment_df = pd.merge(
    sale_df,
    payment_df[["id", "amount"]],
    left_on="payment_id",
    right_on="id",
    how="left",
    suffixes=("_sale", "_payment"),
    validate="one_to_one"
)

print("Sales and payment tables joined successfully!")
print("Joined shape:", sales_payment_df.shape)

Sales and payment tables joined successfully!
Joined shape: (23188, 8)


In [33]:
# Calculate spending features by customer

spending_features = sales_payment_df.groupby(
    "customer_id"
).agg(
    total_spending=("amount", "sum"),
    average_order_value=("amount", "mean"),
    payment_count=("amount", "count")
).reset_index()

print("Spending features created!")

print("Spending dataset shape:", spending_features.shape)

display(spending_features.head())

Spending features created!
Spending dataset shape: (221, 4)


,customer_id,total_spending,average_order_value,payment_count
0,1,74455,3722.750000,20
1,2,1036961,4098.660079,253
2,3,1284692,4225.960526,304
3,4,9577504,9078.202844,1055
4,5,179738,4084.954545,44


In [34]:
# Check spending feature quality

print("Missing values:")
print(spending_features.isnull().sum())

print("\nDuplicate customer records:")
print(spending_features["customer_id"].duplicated().sum())

print("\nTotal customers with spending records:")
print(spending_features["customer_id"].nunique())

Missing values:
customer_id            0
total_spending         0
average_order_value    0
payment_count          0
dtype: int64

Duplicate customer records:
0

Total customers with spending records:
221


In [35]:
# Save spending features

spending_features.to_csv(
    rf"C:\Users\Hp\Downloads\P1G7\P1G7\data processed\spending_features.csv",
    index=False
)

print("Spending features saved successfully!")

Spending features saved successfully!


In [36]:
# Inspect product review columns

print("Product review columns:")
print(product_review_df.columns.tolist())

print("\nProduct review data types:")
print(product_review_df.dtypes)

Product review columns:
['id', 'star_rating', 'product_id', 'customer_id', 'sale_id']

Product review data types:
id             int64
star_rating    int64
product_id     int64
customer_id    int64
sale_id        int64
dtype: object


In [37]:
# Calculate review behaviour for each customer

review_features = product_review_df.groupby(
    "customer_id"
).agg(
    total_reviews=("id", "count"),
    average_rating=("star_rating", "mean"),
    minimum_rating=("star_rating", "min"),
    maximum_rating=("star_rating", "max")
).reset_index()

print("Customer review features created!")

print("Review feature shape:", review_features.shape)

display(review_features.head())

Customer review features created!
Review feature shape: (213, 5)


,customer_id,total_reviews,average_rating,minimum_rating,maximum_rating
0,1,10,2.400000,1,5
1,2,113,3.176991,1,5
2,3,134,3.082090,1,5
3,4,537,3.299814,1,5
4,5,18,3.555556,1,5


In [38]:
# Convert CRM contact timestamps

crm_df["recieved_datatime"] = pd.to_datetime(
    crm_df["recieved_datatime"],
    errors="coerce"
)

print("CRM timestamp conversion completed.")

print(
    "Invalid CRM timestamps:",
    crm_df["recieved_datatime"].isna().sum()
)

CRM timestamp conversion completed.
Invalid CRM timestamps: 0


In [39]:
# Aggregate CRM interactions by customer

crm_features = crm_df.groupby("customer_id").agg(
    total_contacts=("customer_id", "size"),
    total_complaints=(
        "type",
        lambda x: (x == "complaint").sum()
    ),
    total_queries=(
        "type",
        lambda x: (x == "query").sum()
    )
).reset_index()

print("CRM interaction features created!")

print("CRM feature shape:", crm_features.shape)

display(crm_features.head())

CRM interaction features created!
CRM feature shape: (200, 4)


,customer_id,total_contacts,total_complaints,total_queries
0,1,3,2,1
1,2,45,21,24
2,3,167,90,77
3,4,281,132,149
4,5,5,1,4


In [40]:
# Count CRM contacts by platform

platform_features = pd.crosstab(
    crm_df["customer_id"],
    crm_df["platform"]
).reset_index()

# Rename the channel columns

platform_features = platform_features.rename(
    columns={
        "['e-mail']": "email_contacts",
        "['telephone']": "telephone_contacts",
        "['live-chat']": "live_chat_contacts"
    }
)

print("CRM platform features created!")

display(platform_features.head())

CRM platform features created!


platform,customer_id,email_contacts,live_chat_contacts,telephone_contacts
0,1,1,0,2
1,2,28,7,10
2,3,106,24,37
3,4,164,51,66
4,5,2,1,2


In [41]:
# Combine CRM interaction and platform features

crm_customer_features = pd.merge(
    crm_features,
    platform_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

print("Combined CRM features:")
print("Shape:", crm_customer_features.shape)

display(crm_customer_features.head())

Combined CRM features:
Shape: (200, 7)


,customer_id,total_contacts,total_complaints,total_queries,email_contacts,live_chat_contacts,telephone_contacts
0,1,3,2,1,1,0,2
1,2,45,21,24,28,7,10
2,3,167,90,77,106,24,37
3,4,281,132,149,164,51,66
4,5,5,1,4,2,1,2


In [42]:
# Save review features

review_features.to_csv(
    rf"C:\Users\Hp\Downloads\P1G7\P1G7\data processed\review_features.csv",
    index=False
)

# Save CRM features

crm_customer_features.to_csv(
    rf"C:\Users\Hp\Downloads\P1G7\P1G7\data processed\crm_features.csv",
    index=False
)

print("Review and CRM feature files saved successfully!")

Review and CRM feature files saved successfully!


In [43]:
# Convert server log timestamps

server_df["datetime"] = pd.to_datetime(
    server_df["datetime"],
    errors="coerce"
)

print("Server timestamps converted!")

print(
    "Invalid server timestamps:",
    server_df["datetime"].isna().sum()
)

Server timestamps converted!
Invalid server timestamps: 0


In [44]:
# Create a temporary customer email-to-ID mapping

customer_mapping = customer_df[
    ["id", "email"]
].rename(
    columns={"id": "customer_id"}
)

# Attach customer IDs to server log records

server_with_customer = pd.merge(
    server_df,
    customer_mapping,
    left_on="login",
    right_on="email",
    how="left",
    validate="many_to_one"
)

print("Server logs matched to customers!")

print(
    "Unmatched server records:",
    server_with_customer["customer_id"].isna().sum()
)

Server logs matched to customers!
Unmatched server records: 0


In [45]:
# Aggregate server activity by customer

server_features = server_with_customer.groupby(
    "customer_id"
).agg(
    total_login_events=("session_id", "count"),
    unique_sessions=("session_id", "nunique"),
    first_login=("datetime", "min"),
    last_login=("datetime", "max")
).reset_index()

print("Server activity features created!")

print("Server feature shape:", server_features.shape)

display(server_features.head())

Server activity features created!
Server feature shape: (221, 5)


,customer_id,total_login_events,unique_sessions,first_login,last_login
0,1,48,48,2026-01-01 17:28:25.008,2026-01-11 15:41:35.247
1,2,1085,1085,2026-01-01 16:48:25.008,2026-11-09 21:35:20.498
2,3,624,624,2026-01-01 17:40:25.009,2026-11-09 20:43:20.496
3,4,1540,1540,2026-01-01 14:35:05.006,2026-11-09 21:56:40.499
4,5,388,388,2026-01-01 17:51:05.010,2026-04-13 20:45:25.463


In [46]:
# Save server activity features

server_features.to_csv(
    rf"C:\Users\Hp\Downloads\P1G7\P1G7\data processed\server_features.csv",
    index=False
)

print("Server activity features saved successfully!")

Server activity features saved successfully!


In [47]:
# Match survey respondents to customer IDs

survey_with_customer = pd.merge(
    survey_df,
    customer_mapping,
    left_on="email",
    right_on="email",
    how="left",
    validate="one_to_one"
)

print("Survey matching completed!")

print(
    "Unmatched survey respondents:",
    survey_with_customer["customer_id"].isna().sum()
)

Survey matching completed!
Unmatched survey respondents: 0


In [48]:
# Convert survey rating columns to numeric

score_columns = [
    "nps_recommend",
    "satisfaction_score",
    "online_comparison_freq"
]

for column in score_columns:
    survey_with_customer[column] = pd.to_numeric(
        survey_with_customer[column],
        errors="coerce"
    )

print("Survey scores converted to numeric values!")

print(
    survey_with_customer[score_columns].dtypes
)

Survey scores converted to numeric values!
nps_recommend             int64
satisfaction_score        int64
online_comparison_freq    int64
dtype: object


In [49]:
# Check survey score ranges

for column in score_columns:
    print(
        column,
        "minimum:", survey_with_customer[column].min(),
        "maximum:", survey_with_customer[column].max()
    )

nps_recommend minimum: 0 maximum: 10
satisfaction_score minimum: 0 maximum: 10
online_comparison_freq minimum: 0 maximum: 10


In [50]:
# Select survey features without personal information

survey_features = survey_with_customer[
    [
        "customer_id",
        "reported_income",
        "nps_recommend",
        "satisfaction_score",
        "online_comparison_freq"
    ]
].copy()

print("Survey features created!")

print("Survey feature shape:", survey_features.shape)

display(survey_features.head())

Survey features created!
Survey feature shape: (221, 5)


,customer_id,reported_income,nps_recommend,satisfaction_score,online_comparison_freq
0,1,26k-30k,8,6,5
1,2,34k-38k,5,8,9
2,3,26k-30k,9,5,5
3,4,198k-200k+,9,3,5
4,5,42k-46k,0,8,10


In [51]:
# Check survey feature quality

print("Missing values:")
print(survey_features.isnull().sum())

print(
    "\nDuplicate customer IDs:",
    survey_features["customer_id"].duplicated().sum()
)

print(
    "\nUnique survey customers:",
    survey_features["customer_id"].nunique()
)

Missing values:
customer_id               0
reported_income           0
nps_recommend             0
satisfaction_score        0
online_comparison_freq    0
dtype: int64

Duplicate customer IDs: 0

Unique survey customers: 221


In [52]:
# Save survey features

survey_features.to_csv(
    rf"C:\Users\Hp\Downloads\P1G7\P1G7\data processed\survey_features.csv",
    index=False
)

print("Survey features saved successfully!")

Survey features saved successfully!


In [53]:
# Create a clean customer-level starting dataset

base_customer_df = customer_subscription_df[
    [
        "id_customer",
        "start_date",
        "end_date",
        "auto_renew",
        "subscription_type_id",
        "churn"
    ]
].copy()

# Rename customer ID for consistent merging

base_customer_df = base_customer_df.rename(
    columns={"id_customer": "customer_id"}
)

print("Base customer dataset created!")

print("Shape:", base_customer_df.shape)

print("Unique customers:",
      base_customer_df["customer_id"].nunique())

Base customer dataset created!
Shape: (221, 6)
Unique customers: 221


In [54]:
# Prepare subscription type information

subscription_type_features = subscription_type_df[
    ["id", "name", "delivery_service"]
].rename(
    columns={
        "id": "subscription_type_id",
        "name": "subscription_type"
    }
)

# Merge subscription type information

final_df = pd.merge(
    base_customer_df,
    subscription_type_features,
    on="subscription_type_id",
    how="left",
    validate="many_to_one"
)

print("Subscription information added!")
print("Shape:", final_df.shape)

Subscription information added!
Shape: (221, 8)


In [55]:
# Merge sales features

final_df = pd.merge(
    final_df,
    sales_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

# Merge purchase quantity features

final_df = pd.merge(
    final_df,
    quantity_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

# Merge spending features

final_df = pd.merge(
    final_df,
    spending_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

print("Sales and spending information added!")
print("Current dataset shape:", final_df.shape)

Sales and spending information added!
Current dataset shape: (221, 15)


In [56]:
# Add product review features

final_df = pd.merge(
    final_df,
    review_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

# Add CRM interaction features

final_df = pd.merge(
    final_df,
    crm_customer_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

# Add server activity features

final_df = pd.merge(
    final_df,
    server_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

print("Review, CRM and server features added!")
print("Current dataset shape:", final_df.shape)

Review, CRM and server features added!
Current dataset shape: (221, 29)


In [57]:
# Add customer survey information

final_df = pd.merge(
    final_df,
    survey_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

print("Survey information added!")

print("Final combined dataset shape:", final_df.shape)

Survey information added!
Final combined dataset shape: (221, 33)


In [58]:
# Fill missing activity counts with zero

count_columns = [
    "total_orders",
    "total_products_purchased",
    "payment_count",
    "total_reviews",
    "total_contacts",
    "total_complaints",
    "total_queries",
    "email_contacts",
    "telephone_contacts",
    "live_chat_contacts",
    "total_login_events",
    "unique_sessions"
]

# Only fill columns that actually exist

count_columns = [
    col for col in count_columns
    if col in final_df.columns
]

final_df[count_columns] = final_df[count_columns].fillna(0)

print("Missing activity counts handled!")

Missing activity counts handled!


In [59]:
# Check final dataset quality

print("Final dataset shape:")
print(final_df.shape)

print("\nUnique customers:")
print(final_df["customer_id"].nunique())

print("\nDuplicate customer IDs:")
print(final_df["customer_id"].duplicated().sum())

print("\nMissing values by column:")
print(final_df.isnull().sum())

Final dataset shape:
(221, 33)

Unique customers:
221

Duplicate customer IDs:
0

Missing values by column:
customer_id                 0
start_date                  0
end_date                    0
auto_renew                  0
subscription_type_id        0
churn                       0
subscription_type           0
delivery_service            0
total_orders                0
first_order_date            0
last_order_date             0
total_products_purchased    0
total_spending              0
average_order_value         0
payment_count               0
total_reviews               0
average_rating              8
minimum_rating              8
maximum_rating              8
total_contacts              0
total_complaints            0
total_queries               0
email_contacts              0
live_chat_contacts          0
telephone_contacts          0
total_login_events          0
unique_sessions             0
first_login                 0
last_login                  0
reported_income       

In [60]:
# Check the provisional churn distribution

print("Provisional churn distribution:")
print(final_df["churn"].value_counts())

print("\nProvisional churn percentage:")

print(
    (final_df["churn"].value_counts(normalize=True) * 100).round(2)
)

Provisional churn distribution:
churn
0    135
1     86
Name: count, dtype: int64

Provisional churn percentage:
churn
0    61.09
1    38.91
Name: proportion, dtype: float64


In [61]:
# Save combined customer-level dataset

final_df.to_csv(
    rf"C:\Users\Hp\Downloads\P1G7\P1G7\data processed\PerfectPetClub_combined.csv",
    index=False
)

print("Combined customer dataset saved successfully!")

Combined customer dataset saved successfully!


In [62]:
# Count missing values in each column
missing_summary = final_df.isnull().sum()

# Show only columns that contain missing values
missing_summary = missing_summary[missing_summary > 0]

print("Columns with missing values:")
print(missing_summary)

Columns with missing values:
average_rating    8
minimum_rating    8
maximum_rating    8
dtype: int64


In [63]:
print("Dataset data types:")
print(final_df.dtypes)

Dataset data types:
customer_id                          int64
start_date                          object
end_date                            object
auto_renew                           int64
subscription_type_id                 int64
churn                                int64
subscription_type                   object
delivery_service                    object
total_orders                         int64
first_order_date            datetime64[ns]
last_order_date             datetime64[ns]
total_products_purchased             int64
total_spending                       int64
average_order_value                float64
payment_count                        int64
total_reviews                      float64
average_rating                     float64
minimum_rating                     float64
maximum_rating                     float64
total_contacts                     float64
total_complaints                   float64
total_queries                      float64
email_contacts                    

In [64]:
date_columns = [
    "start_date",
    "end_date",
    "first_order_date",
    "last_order_date",
    "first_login",
    "last_login"
]

for column in date_columns:
    if column in final_df.columns:
        final_df[column] = pd.to_datetime(
            final_df[column], errors="coerce"
        )

print("Date columns converted successfully.")

Date columns converted successfully.


In [65]:
# Missing review ratings are retained as NaN.This indicates that these customers have no recorded reviews. We do not replace missing ratings with zero because zero would incorrectly represent an actual rating.

In [66]:
print("Missing values after review-rating check:")
print(final_df[[
    "average_rating",
    "minimum_rating",
    "maximum_rating"
]].isnull().sum())

print("\nFinal dataset shape:", final_df.shape)

Missing values after review-rating check:
average_rating    8
minimum_rating    8
maximum_rating    8
dtype: int64

Final dataset shape: (221, 33)


In [67]:
print("Churn values:")
print(final_df["churn"].value_counts().sort_index())

print("\nChurn percentages:")
print(
    (final_df["churn"].value_counts(normalize=True) * 100)
    .round(2)
)

Churn values:
churn
0    135
1     86
Name: count, dtype: int64

Churn percentages:
churn
0    61.09
1    38.91
Name: proportion, dtype: float64
